# Import Dependencies
Import TensorFlow/Keras and utilities needed for training and saving.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from keras import layers, models
from keras.datasets import mnist
from keras.utils import to_categorical

# Load and Preprocess MNIST
Load MNIST, normalize pixel values, and one-hot encode labels.

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1)).astype("float32") / 255
train_labels = to_categorical(train_labels)

test_images = test_images.reshape((10000, 28, 28, 1)).astype("float32") / 255
test_labels = to_categorical(test_labels)

# Define Drawing Digit CNN
Define the CNN architecture used for drawing digit recognition.

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

model.summary()

# Train and Evaluate
Compile, train, and evaluate the model.

In [ ]:
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(
    train_images,
    train_labels,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Training Curves
Plot accuracy and loss over epochs.

In [ ]:
epochs = range(1, len(history.history["accuracy"]) + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history.history["accuracy"], label="train")
plt.plot(epochs, history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history.history["loss"], label="train")
plt.plot(epochs, history.history["val_loss"], label="val")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

# Export Model
Save the trained model for Backend usage.

In [ ]:
# Colab-friendly export path
if Path("/content").exists():
    output_path = Path("/content") / "models" / "saved_model.h5"
else:
    backend_dir = Path.cwd().parent if Path.cwd().name == "model_training" else Path.cwd()
    output_path = backend_dir / "models" / "saved_model.h5"

output_path.parent.mkdir(parents=True, exist_ok=True)
model.save(output_path.as_posix())
print(f"Saved drawing digit model to: {output_path}")